# Analyse exploratoire — NYPD Crime Data (YTD)

**Objectif :** Valider le pipeline ETL, comprendre la distribution des crimes,
identifier les tendances temporelles, les zones à risque et préparer le modèle DBSCAN.

| Champ | Source |
|---|---|
| Dataset | NYPD Complaint Data Current (YTD) |
| API ID | `5uac-w243` sur NYC Open Data |
| Colonnes utiles | `cmplnt_fr_dt`, `cmplnt_fr_tm`, `boro_nm`, `ofns_desc`, `law_cat_cd`, `latitude`, `longitude` |

In [ ]:
from pathlib import Path
import sys

project_root = Path('..').resolve()
sys.path.append(str(project_root / 'ml'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from nyc_crime_pipeline import (
    DATASET_URL_YTD, LOCAL_YTD,
    load_source_data, clean_crime_data,
    build_time_stats, build_dashboard_stats, build_hotspots,
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

# Charge depuis le CSV local si dispo, sinon tape l'API
SOURCE = LOCAL_YTD if LOCAL_YTD.exists() else DATASET_URL_YTD
print(f'Source : {SOURCE}')

## 1. Chargement et aperçu brut

In [ ]:
raw = load_source_data(SOURCE)
print(f'Shape brut : {raw.shape}')
print(f'Colonnes   : {list(raw.columns)}')
raw.head(3)

In [ ]:
nulls = raw.isna().sum()
null_pct = (nulls / len(raw) * 100).round(1)
null_report = pd.DataFrame({'null_count': nulls, 'null_%': null_pct})
null_report[null_report['null_count'] > 0].sort_values('null_count', ascending=False)

## 2. Nettoyage — données canoniques

In [ ]:
cleaned = clean_crime_data(raw)
print(f'Avant nettoyage : {len(raw):,} lignes')
print(f'Après nettoyage : {len(cleaned):,} lignes')
print(f'Supprimées      : {len(raw) - len(cleaned):,} ({(len(raw)-len(cleaned))/len(raw)*100:.1f}%)')
cleaned.info()
cleaned.head()

## 3. Crimes par borough

In [ ]:
stats = build_dashboard_stats(raw)

by_borough = stats['by_borough']
print(by_borough.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(by_borough['borough'], by_borough['crime_count'], color='steelblue')
ax.bar_label(bars, fmt='{:,.0f}', padding=4)
ax.set_xlabel('Nombre de crimes')
ax.set_title('Crimes par borough (YTD)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Top 20 types de crimes

In [ ]:
by_crime = stats['by_crime_type']
print(by_crime.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(by_crime['crime_type'], by_crime['crime_count'], color='tomato')
ax.bar_label(bars, fmt='{:,.0f}', padding=4)
ax.set_xlabel('Nombre de crimes')
ax.set_title('Top 20 catégories de crimes (YTD)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Répartition par niveau d'infraction (Felony / Misdemeanor / Violation)

In [ ]:
by_law = stats['by_law_category']
print(by_law.to_string(index=False))

fig, ax = plt.subplots(figsize=(5, 4))
ax.pie(
    by_law['crime_count'],
    labels=by_law['law_category'],
    autopct='%1.1f%%',
    startangle=140,
    colors=['#e74c3c', '#e67e22', '#f1c40f']
)
ax.set_title('Répartition Felony / Misdemeanor / Violation')
plt.tight_layout()
plt.show()

## 6. Distribution horaire

In [ ]:
by_hour = stats['by_hour']
print(by_hour.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_hour['hour'], by_hour['crime_count'], color='mediumpurple', width=0.8)
ax.set_xticks(range(0, 24))
ax.set_xlabel('Heure de la journée')
ax.set_ylabel('Nombre de crimes')
ax.set_title('Distribution horaire des crimes (YTD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## 7. Tendances temporelles (par mois)

In [ ]:
by_month = stats['by_month']
# Filtre les mois plausibles (données YTD → après 2020)
by_month = by_month[by_month['month'] >= '2020'].reset_index(drop=True)
print(by_month.tail(12).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(len(by_month)), by_month['crime_count'], marker='o', color='steelblue', linewidth=1.5)
tick_step = max(1, len(by_month) // 18)
ax.set_xticks(range(0, len(by_month), tick_step))
ax.set_xticklabels(by_month['month'].iloc[::tick_step], rotation=45, ha='right')
ax.set_xlabel('Mois')
ax.set_ylabel('Nombre de crimes')
ax.set_title('Évolution mensuelle des crimes (depuis 2020)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## 8. Détection des hotspots — DBSCAN

- `eps_km = 0.3` → rayon de voisinage 300 m
- `min_samples = 15` → minimum 15 crimes pour former un cluster

In [ ]:
hotspots = build_hotspots(cleaned, eps_km=0.3, min_samples=15)
print(f'{len(hotspots)} hotspots détectés')
print(f"Crimes couverts par les clusters : {hotspots['crime_count'].sum():,}")
hotspots.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(
    hotspots['longitude'], hotspots['latitude'],
    s=hotspots['crime_count'] / hotspots['crime_count'].max() * 300 + 10,
    c=hotspots['crime_count'], cmap='YlOrRd', alpha=0.7, edgecolors='grey', linewidths=0.3
)
plt.colorbar(scatter, ax=ax, label='Nombre de crimes')
ax.set_xlim(-74.3, -73.7)
ax.set_ylim(40.5, 40.95)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Hotspots DBSCAN — {len(hotspots)} clusters (eps=300m, min=15)')
ax.set_facecolor('#f0f0f0')
plt.tight_layout()
plt.show()

## 9. Export des données prêtes pour la base PostgreSQL

In [ ]:
output_dir = project_root / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

cleaned.to_csv(output_dir / 'nyc_crime_clean.csv', index=False)
hotspots.to_csv(output_dir / 'nyc_crime_hotspots.csv', index=False)

stats_dir = output_dir / 'stats'
stats_dir.mkdir(exist_ok=True)
for name, df in stats.items():
    df.to_csv(stats_dir / f'{name}.csv', index=False)

print('Exports OK :')
for f in sorted(output_dir.rglob('*.csv')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.relative_to(project_root)}  ({size_kb:.0f} KB)')

## Synthèse

| Indicateur | Valeur |
|---|---|
| Lignes brutes | cf. cellule 1 |
| Lignes nettoyées | cf. cellule 2 |
| Boroughs couverts | 5 |
| Hotspots DBSCAN | cf. cellule 8 |

**Prochaines étapes :**
- Importer `nyc_crime_clean.csv` en base PostgreSQL via `database/schema.sql`
- Brancher le backend FastAPI sur la table `crimes`
- Affiner les paramètres DBSCAN sur le dataset Historic (10M lignes)